# UCMCTrack smoke and camera comparison

Reuse the YOLO handoff and run four independent class trackers over complete sequences. Both camera methods require human review before this notebook can start a real run.


## Environment

Set UA_DETRAC_PROJECT, UA_DETRAC_DATASET, UA_DETRAC_DETECTIONS and UA_DETRAC_CALIBRATION to override paths. Use local Colab storage for repeated image reads.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

# Mount Drive on Colab; local kernels find the repository from parent folders.
try:
    from google.colab import drive, output as colab_output
except ImportError:
    pass
else:
    drive.mount('/content/drive')
    colab_output.enable_custom_widget_manager()

project = os.environ.get('UA_DETRAC_PROJECT')
candidates = ([Path(project)] if project else [Path.cwd(), *Path.cwd().parents,
    Path('/content/drive/MyDrive/CS331/CS331-Vehicle-Tracking')])
ROOT = next((p.resolve() for p in candidates
             if (p / 'scripts/tracking_pipeline.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Set UA_DETRAC_PROJECT to the repository directory.')
sys.path.insert(0, str(ROOT))
local_dataset = Path('/content/ua_detrac/dataset_v1.1')
default_dataset = local_dataset if local_dataset.is_dir() else ROOT / 'outputs/ua_detrac/dataset_v1.1'
DATASET = Path(os.environ.get('UA_DETRAC_DATASET', os.environ.get('UA_DETRAC_OUTPUT', default_dataset)))
DETECTIONS = Path(os.environ.get('UA_DETRAC_DETECTIONS', ROOT / 'outputs/detections_full_yolo26l_E17'))
CALIBRATION = Path(os.environ.get('UA_DETRAC_CALIBRATION', ROOT / 'outputs/tracking/calibration/registry.json'))
RUNS = ROOT / 'outputs/tracking'
METHOD = 'homography'


def run_command(action, output=None, method=None, sequences=None, resume=False, extra=()):
    command = [sys.executable, str(ROOT / 'scripts/tracking_pipeline.py'), action,
        '--dataset', str(DATASET), '--detections', str(DETECTIONS),
        '--calibration', str(CALIBRATION), '--upstream', str(ROOT / 'UCMC/UCMCTrack')]
    if output is not None:
        command += ['--output', str(output)]
    if method is not None:
        command += ['--method', method]
    if sequences:
        command += ['--sequences', *sequences]
    if resume:
        command += ['--resume']
    command += list(extra)
    print('Command:', command)
    return subprocess.run(command, cwd=ROOT, check=True)

print('Dataset:', DATASET)
print('Detections:', DETECTIONS)
print('Calibration:', CALIBRATION)

# Apply the compatibility patch after initializing the pinned dependency.
subprocess.run([sys.executable, str(ROOT / 'scripts/setup_ucmc.py')], cwd=ROOT, check=True)


## Dependencies

Skip the next cell when using the prepared tracking environment.


In [ ]:
# Run once in a new environment. Local users should select the .venv-tracking kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(ROOT / 'requirements-tracking.txt')], cwd=ROOT, check=True)


## Review preflight

The status command exits with code 2 if any camera is pending. Return to notebook 00 and resolve the listed records. Do not edit review statuses just to bypass preflight.


In [ ]:
SMOKE = ['MVI_20012', 'MVI_39811', 'MVI_63521']
for method in ['homography', 'krt']:
    run_command('status', method=method, sequences=SMOKE)


## Run both methods

Tracking covers all frames at 25 FPS. Metrics use declared XML frames only. Inspect overlays and camera_comparison.csv; recommended_method uses HOTA, then IDF1. An exact tie selects homography. This is fixed-detector development evaluation, not official UA-DETRAC PR-MOTA.


In [ ]:
SMOKE_OUTPUT = RUNS / 'smoke'
run_command('smoke', output=SMOKE_OUTPUT, resume=True)


In [ ]:
import pandas as pd
comparison = pd.read_csv(SMOKE_OUTPUT / 'camera_comparison.csv')
display(comparison)
recommendation = json.loads((SMOKE_OUTPUT / 'selected_method_recommendation.json').read_text())
print('Recommended method:', recommendation['recommended_method'])
print('Freeze reviewed camera geometry before full-validation tuning.')
